# Step 4 — ESIM on `overture_case2s`

**Why:** the word2vec baseline (`01_word2vec.ipynb`) averages each address into one vector, so it cannot tell `MAIN ST` from `MAPLE ST` (case2s AUC ≈ 0.6). ESIM lets the two addresses look at each other's words before the comparison.

**Spec (2026-11-02):** frozen word2vec (200-d, skip-gram, 50 epochs) with learned `UNK` and masked `PAD` → BiLSTM (200) → soft alignment both ways (PAD masked with −∞) → `[a; ã; a−ã; a⊙ã]` → composition BiLSTM (200) → masked mean + max pooling → `[v_a; v_b]` → MLP (200, dropout 0.3) → match probability. Training protocol identical to Step 3 (BCE, Adam 1e-3, `is_better` with the case1 guardrail, patience 3, max 20 epochs). Symmetry: original `[v_a; v_b]`, swap test later.

**Build plan:** all layers first, one shape pass through the whole model, then three tests (PAD mask, softmax axis, tiny-set overfit).

## 0. Setup
**Why:** one place for imports and the constants every cell uses.

In [1]:
# Setup: imports and constants
%load_ext autoreload
%autoreload 2

import copy
import os
import re
import sys
import time
from collections import Counter
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset, Subset
from gensim.models import FastText, Word2Vec

from helpers.pairs import load_pairs
from helpers.check_model_performance import review_typical_slices, find_random_sample_given_mask
from siamese_model.data_plumbing import PairDataset   # word-agnostic: returns raw strings
from siamese_model.encoder_step import BiLSTM          # pack -> LSTM -> unpack, unit-agnostic
from siamese_model.training_step import train_epoch, predict   # same batch format as our collate
from siamese_model.real_training import evaluate, is_better, broken_run_problems   # Step 3's per-slice val row, early-stopping rule, health check
from esim.tokenizer import tokenize   # same tokenisation word2vec was trained with

DATASET = "overture_case2s"
EMB_DIM, HIDDEN, MLP_HIDDEN, DROPOUT = 200, 200, 200, 0.3
BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE = 64, 1e-3, 20, 3
THRESHOLD, CASE1_GUARDRAIL = 0.5, 98.0   # same as siamese_model/real_training.py
PAD_ID, UNK_ID = 0, 1

# Embedding setting: the one switch between the ESIM runs (2026-11-05; fastText folded in 2026-11-06)
#   W2V_KIND: "word2vec" or "fasttext" (fastText = word2vec + char 3-6-gram vectors)
#   W2V_MIN_COUNT: 2 = original run (rare house numbers -> UNK), 1 = every word gets a vector
#   runs so far: word2vec mc2 -> esim_mc2/, word2vec mc1 -> esim_mc1/, fasttext mc1 -> esim_ft_mc1/
W2V_KIND = "word2vec"
W2V_MIN_COUNT = 2
W2V_PATH = f"checkpoints/{W2V_KIND}_50_mc{W2V_MIN_COUNT}.model"      # from 01_word2vec.ipynb
CHECKPOINT_DIR = "checkpoints/esim_" + ("ft_" if W2V_KIND == "fasttext" else "") + f"mc{W2V_MIN_COUNT}"   # runs + saved predictions (git-ignored)


pd.set_option("display.width", 250)
pd.set_option("display.expand_frame_repr", False)

## 1. Load the pairs
**Why:** same pair files and shared schema as word2vec and Step 3.
**Does:** loads train / val / test. Test stays untouched until the final evaluation.
**Output:** `train`, `val`, `test` DataFrames and their label arrays

In [2]:
# Load: train / val / test pairs
train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
print(len(train), len(val), len(test))

202568 41865 43844


## 2. The word2vec model
**Why:** ESIM's word vectors come from the 50-epoch word2vec model. `01_word2vec.ipynb` never saved its model, and a retrained one differs run to run (Python's string hashing is randomised per kernel start), so it should be trained once and saved.
**Does:** gets the trained model into this notebook (gensim can save a model to a file and load it back; where the training and saving happen is your call).
**Output:** `w2v`, a trained gensim model with 200-d vectors

In [3]:
# word2vec model: train once and save / load
EMB_DIM = 200
w2v = (FastText if W2V_KIND == "fasttext" else Word2Vec).load(W2V_PATH)   # set in Setup
assert w2v.wv.vector_size == EMB_DIM
print(w2v.epochs, len(w2v.wv.key_to_index))

50 4282


## 3. Vocabulary and embedding matrix
- **Background**: Even though our `word2vec` model technically does the id-encoding as well as vector-encoding, we will split it up into two stage. `ID-Conversion` and `Vector-Conversion`. `Vector-Conversion` happens in this layer. `ID-Conversion` occurs in vocab
- Before we input anything, our `DataLoader` will upload all word_sequences as a series of **id's**, not words
- Then: in our embedding layer, we will take the embedding table from our trained `word2vec` model and place it into `nn.Embedding`. 
- **A Note on nn.Embeddings**: We often think of embeddings as being similar to a dict, but it's more like a list. If we give it a list of embeddings, the key will be the `index` and the value will be the embedding itself. So when we initially add two `zero-embeddings`, it essentially assign `0,1` to a zero embedding. 
- **What are we training on?**: We want our `word2vec` embeddings to be frozen, but we want to have a trainable `<unk>` parameter, because we any arbitrary embedding, will likely implicitly mean something on the embedding space if not trained. We get both by creating an `Embedding Layer` whose **weights are frozen**, which avoids updating gradients, while assigning `<unk>` using `Parameter` to allow only `<unk>` embeddings to be updated

##### Forward Dorward
- When we initially run our id through the embedding layer we get `out` of size $(n, m, h)$
- Our `is_unk` creates a mask of size $(n, m, 1)$, and essentially serves as a boolean mask
- `self.unk` is of size $(h)$, but when used will be broadcasted to size $(n,m,h)$
- when we use `torch.where`, 
    - `is_unk`, serves as the boolean, of which value should we use. 
    - `self.unk` is used as a constant value if `1`
    - `out` serves as a `mapping_function`
    - The reason `self.unk` and `out` work differently is because of their sizes (partially due to broadcasting), affect their output

In [4]:
# Vocab with PAD / UNK, embedding matrix
vocab = {"PAD": PAD_ID,"UNK": UNK_ID, **{k:v+2 for k,v in w2v.wv.key_to_index.items()}}

class EmbeddingLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = w2v
        matrix = torch.cat([torch.zeros(2, EMB_DIM), torch.tensor(w2v.wv.vectors)])
        self.frozen = nn.Embedding.from_pretrained(matrix, freeze = True)
        self.unk = nn.Parameter(torch.zeros(EMB_DIM))
        

    def forward(self, ids):
        out = self.frozen(ids)
        is_unk = (ids == UNK_ID).unsqueeze(-1)
        out = torch.where(is_unk, self.unk, out)
        return out

Embedding checks (written by Claude, 2026-11-04): each word gets its own word2vec row (the +2 shift), `PAD` gives zeros, `UNK` gives the learned `unk`. `unk` is set to ones first, because at init it equals the zero `PAD` row and the `UNK` check couldn't tell them apart.


In [5]:
# Embedding checks: word -> its own word2vec row, PAD -> zeros, UNK -> unk
emb = EmbeddingLayer()
with torch.no_grad():
    emb.unk.fill_(1.0)

n_words = len(w2v.wv.index_to_key)
words = [w2v.wv.index_to_key[i] for i in (0, 1, n_words // 2, n_words - 1)]   # first, second, middle, last
for w in words:
    assert torch.equal(emb(torch.tensor([[vocab[w]]]))[0, 0], torch.tensor(w2v.wv[w])), w
assert (emb(torch.tensor([[PAD_ID]])) == 0).all()
assert torch.equal(emb(torch.tensor([[UNK_ID]]))[0, 0], emb.unk)
print("embedding checks passed on", words)


embedding checks passed on ['ST', 'RD', '1096', '1586']


## 4. Encode a batch
- We just tokenize our variables
- We have to minimize/limit size to avoid batch-size blowup (This more of concern when dealing with `chars`, but not so much for words)
- We pad it so they can be turned into a 2D tensor


In [6]:
# encode_batch (from siamese_model/encode_batch_step.py, now at word level)
MAX_BATCH_FEATURE_SIZE = 85
def encode_batch(strings, vocab, max_len = MAX_BATCH_FEATURE_SIZE):
    strings = [tokenize(s)[:max_len] for s in strings]
    batch_len = max([len(s) for s  in strings])
    ret = []
    for tokens in strings:
        res = [vocab.get(token, UNK_ID) for token in tokens]
        res += [PAD_ID] * (batch_len - len(res))
        ret.append(res)
    lengths = [len(s) for s  in strings]
    return torch.tensor(ret, dtype = torch.long), torch.tensor(lengths, dtype = torch.long)


- We were considering that an empty string could silently break gradients, test shows we have no empty addresses, so this is a non-issue

In [7]:
all_addr = pd.concat([d[c] for d in (train, val, test) for c in ("address_a", "address_b")])
empty = all_addr[all_addr.map(tokenize).str.len() == 0]
print(len(empty)); empty.value_counts().head()


0


Series([], Name: count, dtype: int64)

## 5. Dataset and DataLoaders
**Why:** feeds the model one batch at a time, encoding each batch only when it is drawn.
**Does:** a `Dataset` returning raw address strings and the label, a collate function that encodes both sides, and loaders (train shuffled, val and test not).
**Output:** `train_loader`, `val_loader`, `test_loader`

In [8]:
def make_collate(vocab):
    def collate(samples):
        # samples = [(address_a, address_b, label), ...] -> encode each side separately
        addr_a, addr_b, labels = zip(*samples)
        ids_a, len_a = encode_batch(list(addr_a), vocab)
        ids_b, len_b = encode_batch(list(addr_b), vocab)
        y = torch.tensor(labels, dtype=torch.float32)
        return ids_a, len_a, ids_b, len_b, y
    return collate

# Loaders (PairDataset imported from siamese_model/data_plumbing.py; make_collate stays here so it uses the word-level encode_batch)
train_dataset = PairDataset(train, train_label)
val_dataset = PairDataset(val, val_label)
test_dataset = PairDataset(test, test_label)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle = True, collate_fn = make_collate(vocab))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle = False, collate_fn= make_collate(vocab))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle = False, collate_fn= make_collate(vocab))

## 6. Encode: BiLSTM
- **BiLSTM**: its output_size is called `hidden_size` NOT `OUTPUT_SIZE` because the resultant sequence is just them dragging along the short-term memory at each timestep
- **dropout**: dropout is a little interesting and does not need to specify size, because it's not a layer w/ weights. It's merely a `mask`, whose size is equivelant to the **input_dims**
    - **But can't we infer input size for all things?**: We can, which is why we also have `nn.LazyLinear` which does exactly that. The ouput_size, however, cannot be inferred, and is a design choice.

In [9]:
# Context encoder: dropout -> shared BiLSTM (BiLSTM imported from siamese_model/encoder_step.py)
class ContextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.dropout = nn.Dropout(p=DROPOUT)
        self.bilstm = BiLSTM(input_size=EMB_DIM, hidden=HIDDEN)   # input = word vector size, output 2 * HIDDEN

    def forward(self, vectors, lengths):
        dropout_val = self.dropout(vectors)
        BiLSTM_val = self.bilstm(dropout_val, lengths)
        return BiLSTM_val


## 7. Align: cross attention
- **What does @ do?**: `@` understands that for 3d operations with size (n, m_a, h) @ (n, h, m_b) that `n` is representative of number of rows, which means instead of 3D dot-product, we instead n operations of (m_a, b) x (h, m_b) 
    - This is also why we do `.transpose(1,2)`, which swaps `dim=1` and `dim=2`
    - The result shape is (B, m_a, m_b)
- **How do we create the Intial Mask?**: 
    - When we do B.size(1), we create w/ `torch.arange(B.size(1))`, our output_size is (m_b)
    - Next `len_b` is initially (n), but after `[:, None]` it becomes (n, 1)
    - Now we have to deal with `broadcasting`. After broadcasting:
        - `arange`: (m_b) -> (1, m_b) -> (n, m_b)
        - `len_b`: (n, 1) -> (n, m_b)
        - `result`: (n, m_b)
        - **NOTE**, we initially add 1 to right of `len_b`, cause broadcasting only attempts to add 1s to left
    - We can think of it like a `dict` where each tensor of length `seq_b_length` corresponds to a row in `rows`
- **How do we do Filled Mask?**
    - Firstly we handle mask whose shape is current (n, m_b) becomes (n, 1, m_b)
        - We use `~` to get the inverse because `original_mask` includes all values who are *within the length*
    - Now, we need to update mask to match the size of e. In order to understand, we can think of the initial mask representing a `dict`. That dict says for THIS ROW, we only use these versions of `seq_b`. Then, in order to apply it to the `attention_matrix`, we just need to think, now just apply it and tell all `seq_a` to ignore these values for `seq_b`
    - `softmax`takes the last dimension (which is `dim=2`), and makes it so their sum becomes 1

In [10]:
# Soft alignment with PAD mask
def attention_weights(A, B, len_B):
    # B (rows, seq_b_length, embedding)
    e = A @ B.transpose(1,2) # (rows, seq_a_length, seq_b_length)
    mask = torch.arange(B.size(1), device = B.device) < len_B[:, None] # (rows, seq_b_length)
    return e.masked_fill(~mask[:, None, :], float("-inf")).softmax(dim = -1) # (rows, seq_a_length, seq_b_length)

def align(A, B, len_B):
    w = attention_weights(A, B, len_B)
    return w @ B # (rows, seq_a_length, seq_b_length) x (rows, seq_b_length, embedding) -> (rows, seq_a_length, embedding)


A check to ensure that A_tilde produced by `align` gives the same shape as `original_A`

In [11]:
emb, enc = EmbeddingLayer(), ContextEncoder()
ids_a, len_a, ids_b, len_b, y = next(iter(train_loader))

A = enc(emb(ids_a), len_a)
B = enc(emb(ids_b), len_b)

A_tilde = align(A, B, len_b)
B_tilde = align(B, A, len_a)
assert A_tilde.shape == A.shape and B_tilde.shape == B.shape

We want to ensure that our attention `softmax` actually runs on the correct_dimension (dim = 2).
- This is important cause otherwise `softmax` can succeed while silently running on the wrong axis, messing up our `attention`/model

In [12]:
mask_index = 4
X, Y = torch.randn((1,8, 5)), torch.randn((1,8, 5))   # not A / B: those names hold the real batch used by later checks
len_Y = torch.tensor([mask_index])
res = attention_weights(X, Y, len_Y)
assert torch.allclose(res[:, :, :mask_index].sum(-1), torch.ones(res.shape[:2]), atol = 1e-6)
assert torch.allclose(res[:, :, mask_index:].sum(-1), torch.zeros(res.shape[:2]), atol = 1e-6)

## 8. Enhance
**Why:** puts match and mismatch signals side by side so later layers can read them.
**Does:** per word, concatenates `a`, `ã`, `a − ã` and `a ⊙ ã`.
**Output:** `M_a`, `M_b` (batch × length × 4H)

In [13]:
# Enhancement
def enhance(A, len_A, B, len_B):
    tilde_A = align(A, B, len_B)
    tilde_B = align(B, A, len_A)

    A_res = torch.cat([A, tilde_A, A - tilde_A, A * tilde_A], dim = 2)
    B_res = torch.cat([B, tilde_B, B - tilde_B, B * tilde_B], dim = 2)

    return A_res, B_res

1) Check that shape is same size except for embedding which should be 4 times larger
2) first 400 (2 * EMB_DIM) should be `original_A`
3) These tensors have `require_grad`, meaning they can be linked back to the model

In [14]:
M_a, M_b = enhance(A, len_a, B, len_b)
assert M_a.shape == (*A.shape[:2], 4 * A.size(-1))       # (n, m_a, 1600)
assert torch.equal(M_a[..., :400], A)                    # first block is A itself
assert M_a.requires_grad                                 # gradient history kept


## 9. Compose: second BiLSTM
**Why do we only process one side?**: We only process one side because later inside the `ESIM Module`, we will run `enhance` and run `Composition` per word_sequence then
- Add `relu` to avoid running a simple linear function

In [15]:
# Composition BiLSTM

class Composition(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4*2*HIDDEN, HIDDEN)   # [a; ã; a−ã; a⊙ã] of 2*HIDDEN each -> HIDDEN
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(DROPOUT)
        self.BiLSTM = BiLSTM(HIDDEN, HIDDEN)

    def forward(self, vectors, lengths):
        linear_val = self.linear(vectors)
        relu_val = self.relu(linear_val)
        dropout_val = self.dropout(relu_val)
        BiLSTM_val = self.BiLSTM(dropout_val, lengths)
        return BiLSTM_val


- Check that the output embedding size matches twice the embedding dimension
- Make sure that BiLSTM padded comparisons are not computed and are `0`

In [16]:
comp = Composition()
C_a = comp(M_a, len_a)
C_b = comp(M_b, len_b)
assert C_a.shape == (*M_a.shape[:2], HIDDEN * 2)
assert (C_a[0, len_a[0]:] == 0).all()

## 10. Pool
**Output Notes**: 
- (B, 800), since word_sequences get compressed
- concatenated `mean/max` into same embedding

In [17]:
# Masked mean + max pooling
class MeanMaxPooling(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, h, lengths):
        lengths = lengths.to(h.device)
        mask = torch.arange(h.size(1), device=h.device)[None, :] < lengths[:, None]   # (B, T), True = real position
        m = mask.unsqueeze(-1)  # (B, T, 1), broadcasts over features
        mean_pool = (h * m).sum(1) / m.sum(1) # pads add 0 and don't count: (B, T, D) -> (B, D)
        max_pool = h.masked_fill(~m, float("-inf")).max(dim=1).values
        return torch.cat([mean_pool, max_pool], dim = 1)

In [18]:
pooling = MeanMaxPooling()
v_a, v_b = pooling(C_a, len_a), pooling(C_b, len_b)
assert v_a.shape == (C_a.size(0), 4 * HIDDEN)                     # mean + max of 2 * HIDDEN each
assert torch.isfinite(v_a).all() and torch.isfinite(v_b).all()   # a -inf from the max mask would show here

## 11. Head and the assembled model
- Pretty Standard just squash both layers on top of each other before sending to `ClassifierHead`

In [19]:
class ClassifierHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(HIDDEN * 4 * 2, MLP_HIDDEN)   # [v_a; v_b], 4 * HIDDEN each
        self.tanh = nn.Tanh()
        self.dropout = nn.Dropout(DROPOUT)
        self.linear2 = nn.Linear(MLP_HIDDEN, 1)
        

    def forward(self, vectors):
        linear1_val = self.linear1(vectors)
        tanh_val = self.tanh(linear1_val)
        dropout_val = self.dropout(tanh_val)
        linear2_val = self.linear2(dropout_val)
        return linear2_val.squeeze(-1)

# ESIM model
class ESIM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = EmbeddingLayer()
        self.context_encoder = ContextEncoder()
        # attribute names are the state_dict keys of the saved checkpoints: renaming one breaks load_state_dict
        self.encode_attention = Composition()
        self.pooling = MeanMaxPooling()
        self.head = ClassifierHead()

    def forward(self, ids_a, len_a, ids_b, len_b):
        embedding_a = self.embedding(ids_a)
        embedding_b = self.embedding(ids_b)

        context_a = self.context_encoder(embedding_a, len_a)
        context_b = self.context_encoder(embedding_b, len_b)

        A_res, B_res = enhance(context_a, len_a, context_b, len_b)

        composed_a = self.encode_attention(A_res, len_a)
        composed_b = self.encode_attention(B_res, len_b)

        pooling_a = self.pooling(composed_a, len_a)
        pooling_b = self.pooling(composed_b, len_b)

        final_vector = torch.cat([pooling_a, pooling_b], dim = 1)        
        return self.head(final_vector)



Check that the classifier_head, generates a 1D tensor whose length equals number of rows

In [20]:
head = ClassifierHead()
x = torch.cat([v_a, v_b], dim = -1)
logits = head(x)
assert logits.shape == (x.size(0),)

In [21]:
esim = ESIM()
ids_a, len_a, ids_b, len_b, y = next(iter(train_loader))
res = esim(ids_a, len_a, ids_b, len_b)
assert res.shape == (ids_a.size(0),)

## 12. One pass and tests
**Why:** shape bugs crash, but a missing PAD mask or a wrong softmax axis only makes the numbers worse, so those need explicit checks.
**Does:** (a) one batch through the whole model: the shape assert at the end of §11; (b) mask test: the same pair padded to two different lengths gives the same output; (c) axis check: each row of the attention weights sums to 1 — moved to §7; (d) overfit test: a tiny set reaches near-zero loss.
**Output:** passing asserts

#### B) Mask Test
Testing if our end-to-end code can properly filter out padding tokens
- make sure to set to `eval` to not use `dropout`. By default starts in `train` mode
- use `torch.allclose` to compare since dot-products will not return exact result

In [22]:
# Shape pass and tests

def modified_ids(ids, lengths):
    mask = torch.arange(ids.size(1)) >= lengths[:, None]   # position == length is the first PAD
    new_ids = torch.where(mask, 2, ids)
    return new_ids

# Testing Masking End-to-End
orig_ids = [[2,0,0], [2,2,0], [1,2,3]]
orig_lens = [1,2,3]

esim = ESIM().eval()
ids_a, len_a = torch.tensor(orig_ids), torch.tensor(orig_lens)
ids_b, len_b = torch.tensor([[3,4,5], [4,0,0], [5,3,0]]), torch.tensor([3,1,2])   # different lengths from A, so a len_a/len_b swap would fail
mod_a, mod_b = modified_ids(ids_a, len_a), modified_ids(ids_b, len_b)

assert torch.allclose(esim(ids_a, len_a, ids_b, len_b), esim(mod_a, len_a, mod_b, len_b), atol = 1e-6)


#### D) Overfit Test
- Unlike in normal training, our goal is to overfit the set
- This is so we're able to test the gradients of our model and assert the MODEL CAN LEARN

In [23]:
toy = pd.DataFrame([
    ("12 main st",        "12 main st",         1),   # identical
    ("12 main st",        "12 main street",     1),   # abbreviation
    ("45 oak avenue",     "45 oak ave",         1),   # abbreviation
    ("301 elm road",      "301 elm raod",       1),   # typo
    ("88 pine st apt 2",  "88 pine st",         1),   # extra unit (building-level match)
    ("12 main st",        "14 main st",         0),   # case1: same street, different number
    ("45 oak avenue",     "47 oak avenue",      0),   # case1
    ("301 elm road",      "301 elm street",     0),   # case2-style: same number, different street type
    ("12 main st",        "990 harbor view dr", 0),   # easy
    ("88 pine st",        "5 quarry lane",      0),   # easy
], columns=["address_a", "address_b", "label"])
toy_labels = toy["label"]

toy_ds = PairDataset(toy, toy_labels)
toy_loader = DataLoader(toy_ds, batch_size=len(toy_ds), shuffle=False, collate_fn=make_collate(vocab))   # one batch = the whole toy set

#TEST: Checks to make sure our dataset is not accidentally filled with words that work is <unk> on encoder
[w for s in pd.concat([toy.address_a, toy.address_b]) for w in tokenize(s) if w not in vocab]

['APT', 'RAOD']

In [24]:
model = ESIM()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
for epoch in range(300):
    loss = train_epoch(model, toy_loader, opt, loss_fn)
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d} loss {loss:.4f}")
assert loss < 0.05

epoch   0 loss 0.6918


epoch  50 loss 0.0000


epoch 100 loss 0.0000


epoch 150 loss 0.0075


epoch 200 loss 0.0000


epoch 250 loss 0.0000


#### E) Frozen vs learned embedding 
- One training step must leave the word2vec rows unchanged and move `unk`
- Needs `UNK` words in the batch, or `unk` gets no gradient: the toy set has `APT` and `RAOD`


In [25]:
# One training step: word2vec rows unchanged, unk moved
model = ESIM()
frozen_before = model.embedding.frozen.weight.clone()
unk_before = model.embedding.unk.detach().clone()
opt = torch.optim.Adam(model.parameters(), lr=LR)
train_epoch(model, toy_loader, opt, nn.BCEWithLogitsLoss())
assert torch.equal(model.embedding.frozen.weight, frozen_before), "word2vec rows changed: not frozen"
assert not torch.equal(model.embedding.unk.detach(), unk_before), "unk did not move: no gradient reached it"


## 13. Training
- Timing and Building training loop

In [26]:
SEED = 0
N_TIMING_BATCHES = 200

torch.manual_seed(SEED)
timing_model = ESIM()
timing_opt = torch.optim.Adam(timing_model.parameters(), lr=LR)
subset_loader = DataLoader(Subset(train_dataset, range(N_TIMING_BATCHES * BATCH_SIZE)), batch_size=BATCH_SIZE, collate_fn = make_collate(vocab))

# train/val loaders come from §2 (train shuffled); don't rebuild them here

t0 = time.perf_counter()
train_epoch(timing_model, subset_loader, timing_opt, nn.BCEWithLogitsLoss())
train_seconds = time.perf_counter() - t0
print(f"train: {train_seconds / N_TIMING_BATCHES:.3f} s/batch -> {train_seconds / N_TIMING_BATCHES * len(train_loader) / 60:.1f} min/epoch")

t0 = time.perf_counter()
predict(timing_model, val_loader)
print(f"val pass: {time.perf_counter() - t0:.1f} s")

train: 0.053 s/batch -> 2.8 min/epoch


val pass: 11.2 s


In [27]:
def run_training(seed=SEED):
    torch.manual_seed(seed)
    model = ESIM()
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr = LR)
    loss_fn = nn.BCEWithLogitsLoss()

    best, best_state, epochs_since_best = None, None, 0
    history = []
    for epoch in range(MAX_EPOCHS):
        train_loss = train_epoch(model, train_loader, opt, loss_fn)

        row = {"epoch": epoch, "train_loss": train_loss, **evaluate(model, val_loader, val, val_label)}
        print(row)
        history.append(row)

        if is_better(row, best):
            best, best_state, epochs_since_best = row, copy.deepcopy(model.state_dict()), 0
        else:
            epochs_since_best += 1
            if epochs_since_best >= PATIENCE:
                break
    if best is None:                                      # no epoch reached the case1 guardrail -> keep the last epoch's weights
        best_state = copy.deepcopy(model.state_dict())
    return pd.DataFrame(history), best, best_state

- training in question

In [28]:
# CHECKPOINT_DIR comes from Setup (one folder per W2V_KIND + W2V_MIN_COUNT); existing runs are skipped
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for seed in [0, 1, 2]:
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"esim_seed_{seed}.pt")
    if os.path.exists(ckpt_path):
        continue

    history, best, best_state = run_training(seed = seed)

    history.to_csv(os.path.join(CHECKPOINT_DIR, f"history_seed_{seed}.csv"), index = False)
    torch.save({
        "state_dict": best_state, "vocab": vocab,
        "config": {"emb_dim": EMB_DIM, "hidden": HIDDEN, "mlp_hidden": MLP_HIDDEN, "dropout": DROPOUT,
                   "lr": LR, "max_len": MAX_BATCH_FEATURE_SIZE, "dataset": DATASET,
                   "w2v_min_count": W2V_MIN_COUNT, "w2v_kind": W2V_KIND},
        "seed": seed, "best": best}, ckpt_path)
    print(f"finishing seed:{seed} -> best: {best}")
    print(f"problems: {broken_run_problems(history)}")


## 14. Results per slice
**Why:** the headline question is whether alignment fixes case2s.
**Does:** val per slice against the word2vec baseline and the Step 3 siamese model; test once, at the end, for the best configuration; row added to `results.md`.
**Output:** slice table

### 14a. Val, per seed 
- Only real concern was the low match rate for positive:identicals, where they were unusually low
    - Also, seed 2 surprising got `100%` right somehow
- This leads us to figuring out that Word2Vec is mishandling similar numbers, where it isn't able to effectively tell the difference between addresses where only the house_number is potentially different

In [29]:
# 14a: val per seed
res_val = {}
for seed in [0, 1, 2]:
    print(f"{'='*20} seed:{seed} {'='*20}")
    ckpt = torch.load(os.path.join(CHECKPOINT_DIR, f"esim_seed_{seed}.pt"))

    model = ESIM()
    model.load_state_dict(ckpt["state_dict"])
    # res = evaluate(model, test_loader, ot_test, ot_test_label)
    # print(res)
    with torch.no_grad():
        val_preds = (predict(model, val_loader) >= THRESHOLD).int().numpy()
        res_val[seed] = val_preds
        np.save(os.path.join(CHECKPOINT_DIR, f"preds_val_seed_{seed}.npy"), val_preds)   # read by 14d
        val_slices = review_typical_slices(val, val_preds, val_label)
        print(val_slices)

==================== seed:0 ====================


                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3341      49     98.6%
1                  positives: non-identical    797      659     138     82.7%
2     neg: case1 (same street, diff number)  13781    13545     236     98.3%
3  neg: case2 (same number, similar street)     34       19      15     55.9%
4                         neg: case3 (easy)  22969    22951      18     99.9%
5   neg: case2s (swapped look-alike street)    894      680     214     76.1%
6                        overall (accuracy)  41865    41195     670     98.4%
==================== seed:1 ====================


                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3320      70     97.9%
1                  positives: non-identical    797      652     145     81.8%
2     neg: case1 (same street, diff number)  13781    13654     127     99.1%
3  neg: case2 (same number, similar street)     34       19      15     55.9%
4                         neg: case3 (easy)  22969    22952      17     99.9%
5   neg: case2s (swapped look-alike street)    894      713     181     79.8%
6                        overall (accuracy)  41865    41310     555     98.7%
==================== seed:2 ====================


                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3390       0    100.0%
1                  positives: non-identical    797      645     152     80.9%
2     neg: case1 (same street, diff number)  13781    13528     253     98.2%
3  neg: case2 (same number, similar street)     34       19      15     55.9%
4                         neg: case3 (easy)  22969    22959      10    100.0%
5   neg: case2s (swapped look-alike street)    894      715     179     80.0%
6                        overall (accuracy)  41865    41256     609     98.5%


### 14b. Test, all 3 seeds
- Same problem as val, but problem is exacerbated.

In [30]:
res_test = {}
for seed in [0, 1, 2]:
    print(f"{'='*20} seed:{seed} {'='*20}")
    ckpt = torch.load(os.path.join(CHECKPOINT_DIR, f"esim_seed_{seed}.pt"))

    model = ESIM()
    model.load_state_dict(ckpt["state_dict"])
    # res = evaluate(model, test_loader, ot_test, ot_test_label)
    # print(res)
    with torch.no_grad():
        test_preds = (predict(model, test_loader) >= THRESHOLD).int().numpy()
        res_test[seed] = test_preds
        np.save(os.path.join(CHECKPOINT_DIR, f"preds_test_seed_{seed}.npy"), test_preds)   # read by 14d
        test_slices = review_typical_slices(test, test_preds, test_label)
        print(test_slices)

==================== seed:0 ====================


                                      slice      n  correct  errors % correct
0                      positives: identical   3543     3397     146     95.9%
1                  positives: non-identical    843      680     163     80.7%
2     neg: case1 (same street, diff number)  14448    14200     248     98.3%
3  neg: case2 (same number, similar street)     15        6       9     40.0%
4                         neg: case3 (easy)  23942    23916      26     99.9%
5   neg: case2s (swapped look-alike street)   1053      781     272     74.2%
6                        overall (accuracy)  43844    42980     864     98.0%
==================== seed:1 ====================


                                      slice      n  correct  errors % correct
0                      positives: identical   3543     3385     158     95.5%
1                  positives: non-identical    843      657     186     77.9%
2     neg: case1 (same street, diff number)  14448    14224     224     98.4%
3  neg: case2 (same number, similar street)     15        7       8     46.7%
4                         neg: case3 (easy)  23942    23928      14     99.9%
5   neg: case2s (swapped look-alike street)   1053      795     258     75.5%
6                        overall (accuracy)  43844    42996     848     98.1%
==================== seed:2 ====================


                                      slice      n  correct  errors % correct
0                      positives: identical   3543     3542       1    100.0%
1                  positives: non-identical    843      664     179     78.8%
2     neg: case1 (same street, diff number)  14448    13882     566     96.1%
3  neg: case2 (same number, similar street)     15        6       9     40.0%
4                         neg: case3 (easy)  23942    23915      27     99.9%
5   neg: case2s (swapped look-alike street)   1053      778     275     73.9%
6                        overall (accuracy)  43844    42787    1057     97.6%


### 14d. Embedding settings side by side, per slice and seed
- `mc1 (min_count =1)`, did not fix the **house_number** isssues with `positives:identical`, since its effect was not 100%, because some same-street numbers have near-identical vectors
- `ft_mc1` fixed the `positives_identical` issue
- `mc1/mc2` > `ft_mc1` for `case2s`


In [31]:
# 14d: % correct per slice, one column per (setting, seed)  (set up by Claude 2026-11-05; fastText added 2026-11-06)
SETTINGS = {"mc2": "checkpoints/esim_mc2", "mc1": "checkpoints/esim_mc1", "ft_mc1": "checkpoints/esim_ft_mc1"}

def slice_table(split_df, labels, split):
    cols = {}
    for name, folder in SETTINGS.items():
        for seed in [0, 1, 2]:
            path = f"{folder}/preds_{split}_seed_{seed}.npy"
            if os.path.exists(path):
                cols[(name, seed)] = review_typical_slices(split_df, np.load(path), labels).set_index("slice")["% correct"]
    return pd.DataFrame(cols)

for name, split_df, labels in [("val", val, val_label), ("test", test, test_label)]:
    print(f"{'='*20} {name} {'='*20}")
    display(slice_table(split_df, labels, name))


==================== val ====================


mc2                    mc1                  ft_mc1                
                                              0      1       2       0       1       2       0       1       2
slice                                                                                                         
positives: identical                      98.6%  97.9%  100.0%   99.8%   98.8%  100.0%  100.0%  100.0%  100.0%
positives: non-identical                  82.7%  81.8%   80.9%   78.8%   80.3%   81.1%   79.9%   80.2%   81.6%
neg: case1 (same street, diff number)     98.3%  99.1%   98.2%   98.3%   98.7%   98.1%   98.4%   98.9%   98.2%
neg: case2 (same number, similar street)  55.9%  55.9%   55.9%   44.1%   52.9%   50.0%   50.0%   50.0%   50.0%
neg: case3 (easy)                         99.9%  99.9%  100.0%  100.0%  100.0%  100.0%  100.0%  100.0%  100.0%
neg: case2s (swapped look-alike street)   76.1%  79.8%   80.0%   72.7%   83.6%   74.4%   64.3%   71.0%   67.0%
overall (accuracy)                        98.4%  98.7%   98.5%   98.4%   98.7%   98.4%   98.3%   98.6%   98.3%

==================== test ====================


mc2                   mc1                 ft_mc1                
                                              0      1       2      0      1       2       0       1       2
slice                                                                                                       
positives: identical                      95.9%  95.5%  100.0%  99.5%  96.8%  100.0%  100.0%  100.0%  100.0%
positives: non-identical                  80.7%  77.9%   78.8%  76.6%  78.4%   78.4%   77.3%   76.9%   78.4%
neg: case1 (same street, diff number)     98.3%  98.4%   96.1%  97.3%  98.3%   95.7%   98.5%   99.0%   97.7%
neg: case2 (same number, similar street)  40.0%  46.7%   40.0%  26.7%  26.7%   33.3%   33.3%   60.0%   46.7%
neg: case3 (easy)                         99.9%  99.9%   99.9%  99.9%  99.9%   99.9%  100.0%  100.0%   99.9%
neg: case2s (swapped look-alike street)   74.2%  75.5%   73.9%  72.1%  79.6%   74.5%   68.9%   72.1%   70.5%
overall (accuracy)                        98.0%  98.1%   97.6%  97.9%  98.2%   97.5%   98.3%   98.5%   98.1%

## Diagnostics
One-off inspections, kept out of the pipeline cells above. Results are written into the markdown (question → result → query).

### D1. Why do seeds 0 and 1 reject ~150 identical positives?
- We want to figure out why we're dropping rows for `positive:identical` when they're literally the same.
- Results showed it's because the `House_Number` was `<unk>`, which made it incredibly difficult for it to differentiate cases in where `House Number` is different vs same

In [ ]:
# Group = numbers that were UNK in the original run, fixed across settings so mc1 vs mc2 compare the same rows
group_vocab = Word2Vec.load("checkpoints/word2vec_50_mc2.model").wv.key_to_index
for p in [0, 1]:
    preds = res_test[0]   # pick the seed here; the mask and the table must use the same predictions
    mask = (test["address_a"] == test["address_b"]).values & (test_label == 1).values & (preds == p)

    df = find_random_sample_given_mask(test, preds, test_label, mask)

    for check in ["a", "b"]:
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])

    display(df[["address_a", "address_b", "predicted", "label", "address_a_ids", "address_b_ids"]])

,address_a,address_b,predicted,label,address_a_ids,address_b_ids
748,2436 MAIN ST,2436 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
813,4031 MAIN ST,4031 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
817,4008 MAIN ST,4008 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
3756,3123 BERLIN TPKE,3123 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
3858,3440 BERLIN TPKE,3440 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
852,2979 MADISON AVE,2979 MADISON AVE,0,1,"[None, 430, 2]","[None, 430, 2]"
840,4675 MAIN ST,4675 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
809,4175 MAIN ST,4175 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
3734,2691 BERLIN TPKE,2691 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
816,4060 MAIN ST,4060 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"


,address_a,address_b,predicted,label,address_a_ids,address_b_ids
1858,2249 NEW LONDON TPKE,2249 NEW LONDON TPKE,1,1,"[1902, 15, 66, 7]","[1902, 15, 66, 7]"
1606,201 SAYBROOK RD,201 SAYBROOK RD,1,1,"[312, 339, 1]","[312, 339, 1]"
1506,9 BRITTON LN,9 BRITTON LN,1,1,"[82, 2974, 26]","[82, 2974, 26]"
892,201 HIGHLAND AVE,201 HIGHLAND AVE,1,1,"[312, 176, 2]","[312, 176, 2]"
2474,18 KINNEY HOLLOW RD,18 KINNEY HOLLOW RD,1,1,"[135, 2391, 342, 1]","[135, 2391, 342, 1]"
900,89 ROWAYTON AVE,89 ROWAYTON AVE,1,1,"[250, 920, 2]","[250, 920, 2]"
613,3 OLD TOWN PARK RD,3 OLD TOWN PARK RD,1,1,"[62, 23, 118, 18, 1]","[62, 23, 118, 18, 1]"
2302,927 BLUE HILLS AVE,927 BLUE HILLS AVE,1,1,"[1841, 479, 133, 2]","[1841, 479, 133, 2]"
4196,209 MAPLE AVE,209 MAPLE AVE,1,1,"[448, 79, 2]","[448, 79, 2]"
3764,1616 WILLARD AVE,1616 WILLARD AVE,1,1,"[2701, 519, 2]","[2701, 519, 2]"


### D2. Is the `UNK` number the pattern across the whole slice? 
- Expansion on D1, to see if this is a systemic issue
- Results show that for all seeds, and example, when it's **comparing None Numbers** is when there's a result split, and when that's not the case, there is NO RESULT SPLIT


In [33]:
# D2: UNK-number share, missed vs found identical positives
extract_non_number = lambda x: x.str.replace(r"^(\d+\s+)", "", regex= True)   # defined before the loop uses it
for seed in [0, 1, 2]:
    preds = res_val[seed]   # pick the seed here; the mask and the table must use the same predictions

    
    # mask = (val["address_a"] == val["address_b"]).values & (val_label == 1).values & (preds == p)
    mask = (val["address_a"] == val["address_b"]).values

    df = find_random_sample_given_mask(val, preds, val_label, mask, size = None)

    
    # Add Tokenized List 
    for check in ["a", "b"]:
        #Embedded Tokens
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])
        #Starts With Number?
        df[f"address_{check}_num_start"] = df[f"address_{check}"].str.contains(r"^\d+")
        df[f"non_number_{check}"] = extract_non_number(df[f"address_{check}"])

    
    df["compare_none_numbers"] = (
        (df["address_a_ids"].str[0].isna()) &
        (df["address_b_ids"].str[0].isna()) & 
        (df["address_a_num_start"] == True) &
        (df["address_b_num_start"] == True) & 
        (df["non_number_a"] == df["non_number_b"])
    )
    display_list = [
        "address_a", "address_b", 
        "predicted", "label", 
        "address_a_ids", "address_b_ids", 
        "address_a_num_start", "address_b_num_start", 
        "non_number_a","non_number_b", 
        "compare_none_numbers"]

    # display(df[display_list])
    print(f"{'='*20} seed:{seed} {'='*20}")
    display(df.groupby(["compare_none_numbers", "predicted"])["predicted"].size().unstack("predicted").sort_index(ascending = False))

==================== seed:0 ====================

predicted,0,1
compare_none_numbers,,
True,49.0,29.0
False,NaN,3312.0


==================== seed:1 ====================


predicted,0,1
compare_none_numbers,,
True,69,9
False,1,3311


==================== seed:2 ====================


predicted,1
compare_none_numbers,
True,78
False,3312


### D3. Does seed 2 pay for its 100% on case1? (val, case1 negatives)
**Notes (David):**
Since we've concluded that the model struggles to compare addresses when house_number except `<unk>` (other than seed 2), we would expect this to carry over for negatives as well
- As shown, `seed0` and `seed1` seem to be able differentiate quite well.
- In comparison `seed2` categorizes all negative values as 

In [34]:
for seed in [0, 1, 2]:
    preds = res_val[seed]
    mask = (val["neg_type"] == "case1").values & (val_label == 0).values   # D2 had: identical strings

    df = find_random_sample_given_mask(val, preds, val_label, mask, size = None)

    for check in ["a", "b"]:
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])
        df[f"address_{check}_num_start"] = df[f"address_{check}"].str.contains(r"^\d+")
        df[f"non_number_{check}"] = extract_non_number(df[f"address_{check}"])

    df["compare_none_numbers"] = (
        (df["address_a_ids"].str[0].isna()) &
        (df["address_b_ids"].str[0].isna()) &
        (df["address_a_num_start"] == True) &
        (df["address_b_num_start"] == True) &
        (df["non_number_a"] == df["non_number_b"])
    )

    print(f"{'='*20} seed:{seed} {'='*20}")
    display(df.groupby(["compare_none_numbers", "predicted"])["predicted"].size().unstack("predicted").sort_index(ascending = False))


==================== seed:0 ====================


predicted,0,1
compare_none_numbers,,
True,141,20
False,13404,216


==================== seed:1 ====================


predicted,0,1
compare_none_numbers,,
True,158,3
False,13496,124


==================== seed:2 ====================


predicted,0,1
compare_none_numbers,,
True,NaN,161.0
False,13528.0,92.0


### D4, confirmation on test set
- Same Test as D2 + D3, but on the test_dataset
- Confirms the same

In [35]:
# D4: D2 (identical positives) and D3 (case1 negatives) on test (written by Claude 2026-11-05)
# Needs extract_non_number from D2 and res_test from 14b.
def unk_number_table(df_split, labels, preds, mask):
    df = find_random_sample_given_mask(df_split, preds, labels, mask, size = None)
    for check in ["a", "b"]:
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])
        df[f"address_{check}_num_start"] = df[f"address_{check}"].str.contains(r"^\d+")
        df[f"non_number_{check}"] = extract_non_number(df[f"address_{check}"])
    df["compare_none_numbers"] = (
        (df["address_a_ids"].str[0].isna()) &
        (df["address_b_ids"].str[0].isna()) &
        (df["address_a_num_start"] == True) &
        (df["address_b_num_start"] == True) &
        (df["non_number_a"] == df["non_number_b"])
    )
    return df.groupby(["compare_none_numbers", "predicted"])["predicted"].size().unstack("predicted").sort_index(ascending = False)

row_groups = {
    "D2 identical positives (1 = correct)": (test["address_a"] == test["address_b"]).values & (test_label == 1).values,
    "D3 case1 negatives (0 = correct)":     (test["neg_type"] == "case1").values & (test_label == 0).values,
}
for name, mask in row_groups.items():
    for seed in [0, 1, 2]:
        print(f"{'='*10} {name} | seed:{seed} {'='*10}")
        display(unk_number_table(test, test_label, res_test[seed], mask))


========== D2 identical positives (1 = correct) | seed:0 ==========


predicted,0,1
compare_none_numbers,,
True,146.0,32.0
False,NaN,3365.0


========== D2 identical positives (1 = correct) | seed:1 ==========


predicted,0,1
compare_none_numbers,,
True,158.0,20.0
False,NaN,3365.0


========== D2 identical positives (1 = correct) | seed:2 ==========


predicted,0,1
compare_none_numbers,,
True,NaN,178.0
False,1.0,3364.0


... (2 outputs left out)

========== D3 case1 negatives (0 = correct) | seed:1 ==========


predicted,0,1
compare_none_numbers,,
True,387,17
False,13837,207


========== D3 case1 negatives (0 = correct) | seed:2 ==========


predicted,0,1
compare_none_numbers,,
True,NaN,404.0
False,13882.0,162.0


### D5. Health Check + Findings Summary
- Health check: `broken_run_problems` on every saved history
    - Checks if any part of history is `NA`
    - Make sure `first_val_loss` <= 0.693 (otherwise would indicate lack of learning)
        - `-log(0.5) = 0.693` ~ BCE bullshit
    - min_train_loss < 0.30
        - Since a pure guess of False (9:1), gets us a loss of about `0.325`, we want to ensure our BEST result is better than the best guess
        - `always guessing negative on a 1:9 dataset = 0.325 (BCE)`
    - Tests to check directly if it's just guessing no.
- Same `UNK` group as D2–D4 (both numbers unknown to the `min_count=2` vocab, rest of the address identical), counted per setting and seed: identical positives found, case1 negatives rejected, and errors outside the group
- Test is confirmation only; the choice is made on val

In [36]:
# D5: health check + D2/D3 group across settings (written by Claude 2026-11-06)
# Needs SETTINGS (14d), group_vocab (D1), extract_non_number (D2).
for name, folder in SETTINGS.items():
    for seed in [0, 1, 2]:
        h = pd.read_csv(f"{folder}/history_seed_{seed}.csv")
        print(f"{name:7s} seed {seed}: {len(h)} epochs, problems: {broken_run_problems(h)}")

def unk_group(split_df):
    """Both addresses start with a number unknown to the min_count=2 vocab, rest of the address identical."""
    grp = np.ones(len(split_df), dtype=bool)
    for side in ["a", "b"]:
        addr = split_df[f"address_{side}"]
        first_unknown = addr.apply(lambda x: group_vocab.get((tokenize(x) or [None])[0]) is None)
        grp &= first_unknown.values & addr.str.contains(r"^\d+").values
    return grp & (extract_non_number(split_df["address_a"]) == extract_non_number(split_df["address_b"])).values

rows = []
for split, split_df, labels in [("val", val, val_label), ("test", test, test_label)]:
    grp = unk_group(split_df)
    ident = (split_df["address_a"] == split_df["address_b"]).values & (labels == 1).values
    case1 = (split_df["neg_type"] == "case1").values & (labels == 0).values
    for name, folder in SETTINGS.items():
        for seed in [0, 1, 2]:
            p = np.load(f"{folder}/preds_{split}_seed_{seed}.npy")
            rows.append({
                "split": split, "setting": name, "seed": seed,
                "identical UNK: found":   f"{(p[ident & grp] == 1).sum()} / {(ident & grp).sum()}",
                "case1 UNK: rejected":    f"{(p[case1 & grp] == 0).sum()} / {(case1 & grp).sum()}",
                "identical other: missed": (p[ident & ~grp] == 0).sum(),
                "case1 other: errors":     (p[case1 & ~grp] == 1).sum(),
            })
display(pd.DataFrame(rows).set_index(["split", "setting", "seed"]))


mc2     seed 0: 11 epochs, problems: []
mc2     seed 1: 9 epochs, problems: []
mc2     seed 2: 8 epochs, problems: []
mc1     seed 0: 5 epochs, problems: []
mc1     seed 1: 12 epochs, problems: []
mc1     seed 2: 8 epochs, problems: []
ft_mc1  seed 0: 5 epochs, problems: []
ft_mc1  seed 1: 8 epochs, problems: []
ft_mc1  seed 2: 8 epochs, problems: []


identical UNK: found case1 UNK: rejected  identical other: missed  case1 other: errors
split setting seed                                                                                       
val   mc2     0                 29 / 78           141 / 161                        0                  216
              1                  9 / 78           158 / 161                        1                  124
              2                 78 / 78             0 / 161                        0                   92
      mc1     0                 72 / 78            21 / 161                        0                   98
              1                 38 / 78           111 / 161                        1                  125
              2                 78 / 78            12 / 161                        0                  116
      ft_mc1  0                 78 / 78           151 / 161                        0                  213
              1                 78 / 78           151 / 161                        0                  137
              2                 78 / 78           144 / 161                        0                  229
test  mc2     0                32 / 178           381 / 404                        0                  225
              1                20 / 178           387 / 404                        0                  207
              2               178 / 178             0 / 404                        1                  162
      mc1     0               162 / 178           149 / 404                        1                  129
              1                72 / 178           316 / 404                        7                  160
              2               178 / 178            64 / 404                        0                  276
      ft_mc1  0               178 / 178           376 / 404                        0                  194
              1               178 / 178           388 / 404                        0                  124
              2               178 / 178           350 / 404                        0                  278